## Generate Embeddings (text-embedding-3-large)

Generates embeddings for all chunks (abstracts and PDF page-chunks) across all 10 categories using `text-embedding-3-large`, called via `langchain-openai`. This unblocks the within-category k-means clustering step (W3 ground truth) and doubles as the first candidate model for the later embedding-model benchmark.

**Scope for this run**: one model only (`text-embedding-3-large`). Multi-model comparison is a separate later stage, this script is not it, it exists purely to produce embeddings good enough to unblock clustering now.

**What gets embedded**:
- Abstracts: the `text` field (title + abstract combined, per your existing schema)
- PDF chunks: the `text` field (extracted page text from DPRA)

**Resumability**: if an embedding output file for a category already exists locally, it's skipped unless `FORCE_REGENERATE = True`. This avoids re-spending API calls/money on categories you've already embedded.

**Storage note**: embeddings are stored as JSONL (`chunk_id` + `embedding` list) for consistency with the rest of your flat-file pipeline. At 3072 dimensions per vector (text-embedding-3-large's default), this is not the most compact format, a `.npy` or Parquet file would be meaningfully smaller and faster to load. Worth switching later if file size becomes a problem once uploaded to Hugging Face, but JSONL keeps this script simple and consistent with your other data files for now.

### Install dependencies (Colab only, skip if already installed locally)

In [6]:
# Uncomment if running in Colab or a fresh environment
# !pip install langchain-openai huggingface_hub python-dotenv tqdm -q
!pip install langchain-openai tqdm -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configuration

In [1]:
import os

# --- Hugging Face repo (shared corpus) ---
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"
REPO_TYPE = "dataset"

# --- Embedding model ---
EMBEDDING_MODEL = "text-embedding-3-large"

# --- The 10 target categories ---
ALL_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

# --- Local output paths ---
PROJECT_ROOT = ".."
EMBEDDINGS_DIR = os.path.join(PROJECT_ROOT, "data", "embeddings", EMBEDDING_MODEL)
os.makedirs(os.path.join(EMBEDDINGS_DIR, "abstracts"), exist_ok=True)
os.makedirs(os.path.join(EMBEDDINGS_DIR, "pdf_chunks"), exist_ok=True)

# --- Behavior ---
FORCE_REGENERATE = False  # set True to re-embed categories that already have output files
BATCH_SIZE = 100          # chunks per API call, OpenAI allows large batches for embeddings
REQUEST_DELAY_SECONDS = 0.5  # small delay between batches to stay comfortably under rate limits

### Authenticate (OpenAI + Hugging Face)

Requires `OPENAI_API_KEY` in your environment or a `.env` file (loaded via `python-dotenv`, matching the pattern in your existing question-generation notebook). Also confirms Hugging Face auth for pulling the source corpus.

In [4]:
from dotenv import load_dotenv
from huggingface_hub import whoami, login

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found, set it in your .env file or environment"

try:
    user_info = whoami()
    print(f"Hugging Face authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated with Hugging Face yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

Hugging Face authenticated as: Sakhiur


### Set up the embedder

`OpenAIEmbeddings` from `langchain-openai` batches multiple texts per API call automatically when you pass a list to `embed_documents`, so we don't need to hand-roll batching logic beyond chunking our input list into reasonably sized pieces (`BATCH_SIZE`) to keep memory and retry scope manageable.

In [7]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL)

### Helper functions

- `load_category_records`: pulls a category's JSONL from the Hugging Face repo (abstracts or PDF chunks) and parses it.
- `embed_records`: batches texts through the embedder, with retry-on-failure so one bad batch doesn't kill the whole run.
- `save_embeddings`: writes `chunk_id` + `embedding` pairs to a local JSONL file.

In [8]:
import json
import time
from huggingface_hub import hf_hub_download


def load_category_records(category: str, corpus_type: str) -> list[dict]:
    """
    corpus_type: 'abstracts' or 'pdf_chunks'
    Returns a list of parsed JSON records, or an empty list if the file doesn't exist on the Hub
    (expected for categories your collaborator hasn't uploaded yet).
    """
    if corpus_type == "abstracts":
        repo_path = f"abstracts/by_category/{category}.jsonl"
    elif corpus_type == "pdf_chunks":
        repo_path = f"pdf_chunks/{category}.jsonl"
    else:
        raise ValueError(f"Unknown corpus_type: {corpus_type}")

    try:
        local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    except Exception as e:
        print(f"  [SKIP] {repo_path} not found on Hub ({e})")
        return []

    records = []
    with open(local_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"  [WARN] {repo_path} line {line_num}: malformed JSON, skipping")
    return records


def get_chunk_id_and_text(record: dict, corpus_type: str) -> tuple[str, str]:
    """Extracts (chunk_id, text_to_embed) from a record, per corpus_type's schema."""
    if corpus_type == "abstracts":
        return record["id"], record["text"]
    elif corpus_type == "pdf_chunks":
        chunk_id = f"{record['paper_id']}_p{record['page_num']}"
        return chunk_id, record["text"]
    else:
        raise ValueError(f"Unknown corpus_type: {corpus_type}")


def embed_records(records: list[dict], corpus_type: str, max_retries: int = 3) -> list[dict]:
    """Embeds a list of records in batches, returns list of {chunk_id, embedding} dicts."""
    results = []

    chunk_ids = []
    texts = []
    for record in records:
        chunk_id, text = get_chunk_id_and_text(record, corpus_type)
        if not text or not text.strip():
            print(f"  [WARN] {chunk_id}: empty text, skipping")
            continue
        chunk_ids.append(chunk_id)
        texts.append(text)

    for batch_start in range(0, len(texts), BATCH_SIZE):
        batch_ids = chunk_ids[batch_start : batch_start + BATCH_SIZE]
        batch_texts = texts[batch_start : batch_start + BATCH_SIZE]

        for attempt in range(1, max_retries + 1):
            try:
                batch_embeddings = embedder.embed_documents(batch_texts)
                break
            except Exception as e:
                print(f"  [WARN] batch {batch_start}-{batch_start+len(batch_texts)}: attempt {attempt} failed ({e})")
                if attempt == max_retries:
                    print(f"  [ERROR] batch {batch_start}-{batch_start+len(batch_texts)}: giving up after {max_retries} attempts")
                    batch_embeddings = [None] * len(batch_texts)
                else:
                    time.sleep(2 ** attempt)  # exponential backoff

        for cid, emb in zip(batch_ids, batch_embeddings):
            if emb is not None:
                results.append({"chunk_id": cid, "embedding": emb})

        time.sleep(REQUEST_DELAY_SECONDS)

    return results


def save_embeddings(embeddings: list[dict], out_path: str) -> None:
    with open(out_path, "w", encoding="utf-8") as f:
        for record in embeddings:
            f.write(json.dumps(record) + "\n")

### Generate embeddings: abstracts

Loops all 10 categories. Categories not yet uploaded by your collaborator are skipped automatically (reported, not treated as an error). Already-embedded categories are skipped unless `FORCE_REGENERATE = True`.

In [9]:
from tqdm import tqdm

abstract_summary = {}

for category in ALL_CATEGORIES:
    out_path = os.path.join(EMBEDDINGS_DIR, "abstracts", f"{category}.jsonl")

    if os.path.exists(out_path) and not FORCE_REGENERATE:
        print(f"[SKIP] {category} abstracts: already embedded at {out_path}")
        continue

    print(f"\n=== {category} abstracts ===")
    records = load_category_records(category, "abstracts")
    if not records:
        abstract_summary[category] = 0
        continue

    embeddings = embed_records(records, "abstracts")
    save_embeddings(embeddings, out_path)
    abstract_summary[category] = len(embeddings)
    print(f"  {len(embeddings)}/{len(records)} embedded -> {out_path}")

print("\nAbstract embedding summary:")
for category, count in abstract_summary.items():
    print(f"  {category}: {count}")


=== cs.AI abstracts ===


c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sakhiur\.cache\huggingface\hub\datasets--Sakhiur--empirical-rag-paradigm-benchmark. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  7369/7369 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.AI.jsonl

=== cs.LG abstracts ===
  10746/10746 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.LG.jsonl

=== cs.IR abstracts ===
  3076/3076 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.IR.jsonl

=== cs.DB abstracts ===
  1532/1532 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.DB.jsonl

=== cs.SE abstracts ===
  2281/2281 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.SE.jsonl

=== cs.CV abstracts ===
  14163/14163 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.CV.jsonl

=== cs.CL abstracts ===
  12418/12418 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.CL.jsonl

=== cs.NE abstracts ===
  1501/1501 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.NE.jsonl

=== cs.DC abstracts ===
  1517/1517 embedded -> ..\data\embeddings\text-embedding-3-large\abstracts\cs.DC.jsonl

=

### Generate embeddings: PDF chunks

Same pattern as abstracts. Note PDF chunks are page-level, so a single paper contributes multiple chunk_ids (`<paper_id>_p<page_num>`), this is expected and matches your DPRA output granularity.

In [10]:
pdf_summary = {}

for category in ALL_CATEGORIES:
    out_path = os.path.join(EMBEDDINGS_DIR, "pdf_chunks", f"{category}.jsonl")

    if os.path.exists(out_path) and not FORCE_REGENERATE:
        print(f"[SKIP] {category} PDF chunks: already embedded at {out_path}")
        continue

    print(f"\n=== {category} PDF chunks ===")
    records = load_category_records(category, "pdf_chunks")
    if not records:
        pdf_summary[category] = 0
        continue

    embeddings = embed_records(records, "pdf_chunks")
    save_embeddings(embeddings, out_path)
    pdf_summary[category] = len(embeddings)
    print(f"  {len(embeddings)}/{len(records)} embedded -> {out_path}")

print("\nPDF chunk embedding summary:")
for category, count in pdf_summary.items():
    print(f"  {category}: {count}")


=== cs.AI PDF chunks ===
  [WARN] 2101.00058_p38: empty text, skipping
  [WARN] 2101.00058_p39: empty text, skipping
  [WARN] 2201.07474_p1: empty text, skipping
  [WARN] 2204.00288_p3: empty text, skipping
  [WARN] 2204.00288_p31: empty text, skipping
  [WARN] 2204.00288_p69: empty text, skipping
  [WARN] 2204.00288_p71: empty text, skipping
  [WARN] 2204.00288_p73: empty text, skipping
  [WARN] 2204.00288_p75: empty text, skipping
  9385/9394 embedded -> ..\data\embeddings\text-embedding-3-large\pdf_chunks\cs.AI.jsonl

=== cs.LG PDF chunks ===
  [WARN] 2408.07084_p5: empty text, skipping
  [WARN] 2410.20302_p29: empty text, skipping
  [WARN] 2501.01287_p6: empty text, skipping
  [WARN] 2501.01287_p7: empty text, skipping
  [WARN] 2501.01287_p8: empty text, skipping
  [WARN] 2501.01287_p9: empty text, skipping
  [WARN] 2501.01287_p10: empty text, skipping
  [WARN] 2501.01287_p11: empty text, skipping
  9775/9783 embedded -> ..\data\embeddings\text-embedding-3-large\pdf_chunks\cs.LG.j

KeyboardInterrupt: 

### Upload embeddings back to Hugging Face

Stored under `embeddings/<model_name>/abstracts/<category>.jsonl` and `embeddings/<model_name>/pdf_chunks/<category>.jsonl`, so the model used is part of the path, this matters once you generate embeddings from a second model for the actual benchmark comparison later.

In [11]:
from huggingface_hub import HfApi

api = HfApi()

for corpus_type in ["abstracts", "pdf_chunks"]:
    local_dir = os.path.join(EMBEDDINGS_DIR, corpus_type)
    for fname in sorted(os.listdir(local_dir)):
        if not fname.endswith(".jsonl"):
            continue
        local_path = os.path.join(local_dir, fname)
        repo_path = f"embeddings/{EMBEDDING_MODEL}/{corpus_type}/{fname}"

        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            commit_message=f"Upload {EMBEDDING_MODEL} embeddings for {corpus_type}/{fname} (by {user_info['name']})",
        )
        print(f"Uploaded {repo_path}")

Processing Files (1 / 1): 100%|██████████|  469MB /  469MB, 4.50MB/s  
New Data Upload: 100%|██████████|  469MB /  469MB, 4.50MB/s  


Uploaded embeddings/text-embedding-3-large/abstracts/cs.AI.jsonl


Processing Files (1 / 1): 100%|██████████|  789MB /  789MB, 4.49MB/s  
New Data Upload: 100%|██████████|  789MB /  789MB, 4.49MB/s  


Uploaded embeddings/text-embedding-3-large/abstracts/cs.CL.jsonl


Processing Files (1 / 1): 100%|██████████| 96.9MB / 96.9MB, 3.78MB/s  
New Data Upload: 100%|██████████| 96.9MB / 96.9MB, 3.78MB/s  


...

Uploaded embeddings/text-embedding-3-large/pdf_chunks/cs.AI.jsonl


Processing Files (0 / 1): 100%|█████████▉|  431MB /  431MB, 7.42MB/s  
New Data Upload: 100%|██████████|  431MB /  431MB, 7.42MB/s  


Uploaded embeddings/text-embedding-3-large/pdf_chunks/cs.IR.jsonl


Processing Files (1 / 1): 100%|██████████|  622MB /  622MB, 7.12MB/s  
New Data Upload: 100%|██████████|  622MB /  622MB, 7.12MB/s  


Uploaded embeddings/text-embedding-3-large/pdf_chunks/cs.LG.jsonl


### Verify

Confirms dimensionality is consistent (should all be the same for a given model) and reports final counts per category, across both corpora, so you can see at a glance what's ready for clustering versus what's still missing (your collaborator's categories, most likely).

In [12]:
print(f"Model: {EMBEDDING_MODEL}\n")

for corpus_type, summary in [("abstracts", abstract_summary), ("pdf_chunks", pdf_summary)]:
    print(f"{corpus_type}:")
    for category in ALL_CATEGORIES:
        out_path = os.path.join(EMBEDDINGS_DIR, corpus_type, f"{category}.jsonl")
        if not os.path.exists(out_path):
            print(f"  {category}: not embedded (missing source or skipped)")
            continue
        with open(out_path, "r", encoding="utf-8") as f:
            lines = [json.loads(l) for l in f if l.strip()]
        dims = {len(r["embedding"]) for r in lines}
        dim_status = dims.pop() if len(dims) == 1 else f"INCONSISTENT: {dims}"
        print(f"  {category}: {len(lines)} vectors, dim={dim_status}")
    print()

Model: text-embedding-3-large

abstracts:
  cs.AI: 7369 vectors, dim=3072
  cs.LG: 10746 vectors, dim=3072
  cs.IR: 3076 vectors, dim=3072
  cs.DB: 1532 vectors, dim=3072
  cs.SE: 2281 vectors, dim=3072
  cs.CV: 14163 vectors, dim=3072
  cs.CL: 12418 vectors, dim=3072
  cs.NE: 1501 vectors, dim=3072
  cs.DC: 1517 vectors, dim=3072
  cs.CR: 1523 vectors, dim=3072

pdf_chunks:
  cs.AI: 9385 vectors, dim=3072
  cs.LG: 9775 vectors, dim=3072
  cs.IR: 6764 vectors, dim=3072
  cs.DB: not embedded (missing source or skipped)
  cs.SE: not embedded (missing source or skipped)
  cs.CV: not embedded (missing source or skipped)
  cs.CL: not embedded (missing source or skipped)
  cs.NE: not embedded (missing source or skipped)
  cs.DC: not embedded (missing source or skipped)
  cs.CR: not embedded (missing source or skipped)

